# LiveProvider

* **LiveProvider** reads pool state directly from a chain RPC and turns it into a **PoolSnapshot**. 
* From there, **StateTwinBuilder** constructs a usable exchange object
* Every primitive in DeFiPy works against it the same way it works against a **MockProvider** recipe.

In [1]:
from defipy import *

### Quickstart — Uniswap V2

In [2]:
from defipy.twin import LiveProvider, StateTwinBuilder

provider = LiveProvider("https://mainnet.infura.io/v3/*")

# WETH/USDC V2 mainnet pool
snap = provider.snapshot(
    "uniswap_v2:0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
)

lp = StateTwinBuilder().build(snap)

# Same shape as any other twin — every primitive works against it
from defipy import CheckPoolHealth
health = CheckPoolHealth().apply(lp)

lp.summary()

print(f"TVL in WETH: {health.tvl_in_token0:.2f}")
print(f"Spot price:  {health.spot_price:.4f}")

Exchange USDC-WETH (LP)
Reserves: USDC = 9659778.463175, WETH = 4207.218562977011
Liquidity: 201595.63304921912 

TVL in WETH: 19319556.93
Spot price:  0.0004


### Quickstart — Uniswap V3

In [5]:
from defipy.twin import LiveProvider, StateTwinBuilder

provider = LiveProvider("https://mainnet.infura.io/v3/*")

# USDC/WETH V3 3000bps mainnet pool
snap = provider.snapshot(
    "uniswap_v3:0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640"
)

lp = StateTwinBuilder().build(snap)

lp.summary()

# All V3 primitives work — pool health, position analysis, slippage, scenarios
from defipy import CheckPoolHealth
health = CheckPoolHealth().apply(lp)

print(f"V3 fee:   {health.fee_pips} pips")
print(f"TVL ratio: {health.tvl_in_token0 / health.tvl_in_token1:.2e}")

Exchange USDC-WETH (LP)
Real Reserves:   USDC = 179091126.028909, WETH = 77923.73045669132
Gross Liquidity: 3735699.215924917 

V3 fee:   500 pips
TVL ratio: 2.30e+03


In [8]:
from defipy import AnalyzePosition
from defipy.twin import LiveProvider, StateTwinBuilder

# Pull live state from a real Uniswap V2 pool — WETH/USDC mainnet
provider = LiveProvider("https://mainnet.infura.io/v3/*")
snapshot = provider.snapshot(
    "uniswap_v2:0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
)
lp = StateTwinBuilder().build(snapshot)
lp.summary()

# Snapshot carries chain context — block_number, timestamp, chain_id
print(f"Block:   {snapshot.block_number}")
print(f"Reserves: {snapshot.token0_name}={snapshot.reserve0:.2f}, "
      f"{snapshot.token1_name}={snapshot.reserve1:.2f}")

# Run any primitive against the live twin — same call shape as MockProvider
result = AnalyzePosition().apply(
    lp,
    lp_init_amt=1.0,
    entry_x_amt=1000,
    entry_y_amt=3_000_000,
)

print(f"Diagnosis:   {result.diagnosis}")
print(f"Net PnL:     {result.net_pnl:.4f}")
print(f"IL %:        {result.il_percentage:.4f}")
print(f"Current val: {result.current_value:.4f}")

Exchange USDC-WETH (LP)
Reserves: USDC = 9659849.799697, WETH = 4207.188322465708
Liquidity: 201595.65291458525 

Block:   25045547
Reserves: USDC=9659849.80, WETH=4207.19
Diagnosis:   il_dominant
Net PnL:     -6888104591.9295
IL %:        -0.9992
Current val: 95.8339
